# Scrap notebooks

This notebook is just a compilation of scrapy tests to view functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.optimize as opt
import scipy.integrate as integr

import sympy as sym

from tqdm import tqdm
import itertools

In [ ]:
# %matplotlib widget

### Plotting solutions

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.set_xlabel(r'$\Delta_d$')
ax.set_ylabel(r'$\Delta_p^{\text{min}}$')

N_value = 10000
s_value = 2000
M_value = 3

Delta_p_max_minus_min_value = 15
alpha_value = 2
beta_value = 1
gamma_value = 4
delta_value = beta_value * gamma_value / (beta_value + gamma_value)
phi_value = 0.3
f_value = 1
E_value = 1

eps = 0.5
dim = 200
alpha = 0.4

Delta_d_values = np.linspace(-50, 50, dim)
Delta_p_min_values = np.linspace(-50, 50, dim)
Delta_d_values, Delta_p_min_values = np.meshgrid(Delta_d_values, Delta_p_min_values)
F_func = stats.uniform(scale = Delta_p_max_minus_min_value, loc = Delta_p_min_values)
#F_func = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = Delta_p_min_values)
Delta_p_full_values = F_func.ppf(M_value / (M_value + 1))

zone = np.zeros((dim, dim))


SC_a_values = zone.copy()
SC_a_values[-2 * delta_value * N_value / (phi_value * s_value) - Delta_d_values <=0] = np.nan
SC_a_values[Delta_p_min_values - E_value - f_value - Delta_d_values - delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_a_values, label="a", alpha=alpha, color="blue")


SC_b_values = zone.copy()
SC_b_values[M_value * (f_value + E_value - Delta_p_full_values) - Delta_d_values - 2 * delta_value * N_value / (phi_value * s_value) <= 0] = np.nan
SC_b_values[E_value + f_value + Delta_d_values + delta_value * N_value / (phi_value * s_value) - Delta_p_full_values <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_b_values, label="b", alpha=alpha, color="orange")

SC_c_values = zone.copy()
SC_c_values[delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value < Delta_p_min_values] = np.nan
SC_c_values[delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value > Delta_p_full_values] = np.nan
SC_c_values[Delta_d_values + delta_value * N_value / (phi_value * s_value) * (2 - F_func.cdf(delta_value * N_value / (phi_value * s_value) + Delta_d_values + f_value + E_value)) >= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_c_values, label="c", alpha=alpha, color="green")

SC_defgh_values = zone.copy()
SC_defgh_values[M_value * (Delta_p_min_values - f_value - E_value) + Delta_d_values - (M_value + 1) * 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan
SC_defgh_values[Delta_d_values - 2 * delta_value * N_value / ((1 - phi_value) * s_value) <= 0] = np.nan

ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_defgh_values, label="defgh", alpha=alpha, color="red")


SC_i_values = zone.copy()
SC_i_values[-2 * delta_value * N_value /(s_value * phi_value) > Delta_d_values] = np.nan
SC_i_values[2 * delta_value * N_value /(s_value * (1 - phi_value)) < Delta_d_values] = np.nan
SC_i_values[2 * (Delta_p_min_values - E_value - f_value) - Delta_d_values * (1 + phi_value) - 2 * delta_value * N_value / s_value <= 0] = np.nan
ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_i_values, label="i", alpha=alpha, color="purple")


SC_j_values = zone.copy()
SC_j_values[-Delta_d_values + M_value * (f_value + E_value) < M_value * Delta_p_min_values - 2 * delta_value * N_value * (M_value + 1) / ((1 - phi_value) * s_value)] = np.nan
SC_j_values[-Delta_d_values + M_value * (f_value + E_value) > M_value * Delta_p_full_values + 2 * delta_value * N_value / (phi_value * s_value)] = np.nan

N_d_j_lim = np.maximum(0,
                       phi_value * (1 - phi_value) * (2 * N_value / (1 - phi_value) - Delta_d_values * s_value / delta_value) / (M_value + 2 + M_value * phi_value)
                      )

#N_d_j_lim[N_d_j_lim < 0] = np.nan
#N_d_j_lim[N_d_j_lim > N_value / (M_value + 1)] = np.nan

SC_j_values[2 * delta_value * (M_value + 1) / s_value *(-N_value/(1 - phi_value) + (1 + M_value * phi_value) / (phi_value * (1 - phi_value)) * N_d_j_lim)\
            + Delta_d_values\
            + M_value*(F_func.ppf(M_value * N_d_j_lim / N_value) - f_value - E_value)\
            > 0] = np.nan

SC_j_values[np.isnan(2 * delta_value * (M_value + 1) / s_value *(-N_value/(1 - phi_value) + (1 + M_value * phi_value) / (phi_value * (1 - phi_value)) * N_d_j_lim)\
            + Delta_d_values\
            + M_value*(F_func.ppf(M_value * N_d_j_lim / N_value) - f_value - E_value))\
           ] = np.nan



ax.plot_surface(Delta_d_values, Delta_p_min_values, SC_j_values, label="j", alpha=alpha, color="brown")

# case k is to hard to plot (solution should be solved for every point)


ax.view_init(-90, 0, 0)
ax.legend()

In [ ]:
color_legend = {"a": "blue",
                "b": "orange",
                "c": "green",
                "defgh": "red",
                "i": "purple",
                "j": "brown",
                "k": "black",
               }

case_to_color_transform = np.vectorize(lambda x: color_legend[x])

In [ ]:
def recursive_root_finder(f, start_min, start_max, depth, eps = 0.001):
        
    if depth == 0:
        return []
    
    if f(start_min) * f(start_max) > 0:
        middle = (start_max + start_min) / 2
        return recursive_root_finder(f, start_min, middle, depth - 1, eps)\
            + recursive_root_finder(f, middle + eps, start_max, depth - 1, eps)

    
    root = opt.root_scalar(f, bracket=[start_min, start_max], method='bisect')

    if not root.converged:
        middle = (start_max + start_min) / 2
        return recursive_root_finder(f, start_min, middle, depth - 1, eps)\
            + recursive_root_finder(f, middle + eps, start_max, depth - 1, eps)
    
    return [root.root]\
        + recursive_root_finder(f, start_min, root.root - eps, depth - 1, eps)\
        + recursive_root_finder(f, root.root + eps, start_max, depth - 1, eps)

In [ ]:
#recursive_root_finder(lambda x: x**2 - 1, -2, 2, 2, 0.001)

In [ ]:
def solution_phi_fixed(N=N_value,
                       s=s_value,
                       delta=delta_value,
                       phi=phi_value,
                       f=f_value,
                       E=E_value,
                       M=M_value,
                       Delta_d=Delta_d_values,
                       F=stats.uniform(loc=1, scale = 3)):



    def SC(N_d, N_p, N_s):
        return delta / s * (N_s**2 / (1 - phi) + N_d * (N_d + N_p) / phi)\
        + (N - N_p) * (f+E)\
        + N_d * Delta_d\
        + N * integr.quad(lambda x: x * F.pdf(x), F.ppf(0), F.ppf(N_p / N))[0]

    def hessian_determinant(N_d, N_p):
        H00 = 2 * delta / s * (1 / (1 - phi) + 1 / phi)
        H01 = delta / s * (2 / (1 - phi) + 1 / phi)
        H11 = 2 * delta / ((1 - phi) * s) + 1 / (N * F.pdf(F.ppf(N_p / N)))

        det = H00 * H11 - H01**2
        return det

    N_ds = []
    N_ps = []
    N_ss = []
    SCs = []
    cases = []

    # check case a


    if 2 * delta * N / (phi * s) + Delta_d < 0 and \
    delta * N / (phi * s) + f + E + Delta_d < F.ppf(0):
        case_a_accepted = True
        N_d = N
        N_p = 0
        N_s = 0
    else:
        case_a_accepted = False

    if case_a_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("a")
        # add the payments


     # check case b

    N_d = N / (M+1)
    N_p = M*N / (M + 1)
    N_s = 0
    if 2 * delta * N / (phi * s) + Delta_d  + M * F.ppf(N_p / N) < M * (f + E) and \
    delta * N / (phi * s) + f + E + Delta_d > F.ppf(N_p / N):
        case_b_accepted = True
    else:
        case_b_accepted = False

    if case_b_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("b")
        # add the payments

    # check case c

    case_c_accepted = False
    if F.ppf(0) <= delta * N / (phi * s) + f + E + Delta_d <= F.ppf(M / (M+1)):
        N_p = N * F.cdf(delta * N / (phi * s) + Delta_d + f + E)
        N_d = N - N_p
        N_s = 0

        if delta * N / (phi * s) + delta * N_d / (phi * s) + Delta_d < 0:    
            case_c_accepted = True
    

    if case_c_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("c")
        # add the payments


    # check case defgh
    case_defgh_accepted = False
    if Delta_d > 2 * delta * N / ((1 - phi) * s) and \
    Delta_d + M * F.ppf(0) > M * (f + E) + 2 * delta * N * (M + 1) / ((1 - phi) * s):
        case_defgh_accepted = True
        N_d = 0
        N_p = 0
        N_s = N

    if case_defgh_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("defgh")
        # add the payments

    # check case i

    case_i_accepted = False

    if -2 * delta * N / (s * phi) <= Delta_d <= 2 * delta * N / (s * (1 - phi)) and \
    2 * (F.ppf(0) - f - E) - Delta_d * (1 + phi) - 2 * delta * N / s > 0:
        case_i_accepted = True
        N_d = phi * (N - Delta_d / (2 * delta / (s * (1 - phi))))
        N_p = 0
        N_s = N - N_d

    if case_i_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("i")
        # add the payments

    # check case j

    case_j_accepted = False
    
    if M * F.ppf(0) - 2 * delta * N * (M + 1) / ((1 - phi) * s) <= M * (f + E) -Delta_d <= M * F.ppf(M/(M+1)) + 2 * delta * N / (phi * s):
        def to_zero_func(N_d):
            return 2 * delta * (M + 1) / s * (-N / (1 - phi) + (1 + M * phi) / (phi * (1 - phi)) * N_d)\
            + Delta_d + M * (F.ppf(M * N_d / N) - f - E)
               
        potential_N_d = opt.root_scalar(to_zero_func, bracket=[0, N / (M + 1)], method='bisect')
        if not potential_N_d.converged:
            raise ValueError("Root finding did not converge for case j")
        N_d = potential_N_d.root
        N_p = M * N_d
        N_s = N - N_d - N_p

        if delta / s * (-2 * (N - (M + 1) * N_d) / (1 - phi) + N_d * (M + 2) / phi) + Delta_d > 0:
            case_j_accepted = True

    if case_j_accepted:
        N_ds.append(N_d)
        N_ps.append(N_p)
        N_ss.append(N_s)
        SCs.append(SC(N_d, N_p, N_s))
        cases.append("j")
        # add the payments

    # check case k
    case_k_accepted = False

    def to_zero_func_k(N_p):
        return -delta * N / s + delta * N_p / ((1 - phi) * s) * (2 - (1 + phi)**2 / (2 * phi))\
            - Delta_d * (1 + phi) / 2 - f - E + F.ppf(N_p / N)
        
    potential_N_ps = recursive_root_finder(to_zero_func_k, 0, N * M / (M + 1), 5, (N * M / (M + 1)) / 1000000) # with a depth of five, we may still skip some solutions as well as with the precision
    SC_candidate_best = None
    #print(len(potential_N_ps))
    for candidate_N_p in potential_N_ps:
        candidate_N_d = N * phi - candidate_N_p * (1 + phi) / 2 - Delta_d * (1 - phi) * phi * s / (2 * delta)
        candidate_N_s = N - candidate_N_d - candidate_N_p

        if candidate_N_d >= candidate_N_p / M\
        and candidate_N_s >= 0\
        and hessian_determinant(candidate_N_d, candidate_N_p) > 0:
            case_k_accepted = True
            SC_candidate = SC(candidate_N_d, candidate_N_p, candidate_N_s)
            if SC_candidate_best is None or SC_candidate < SC_candidate_best:
                N_d_best = candidate_N_d
                N_p_best = candidate_N_p
                N_s_best = candidate_N_s
                SC_candidate_best = SC_candidate

    if case_k_accepted:
        N_ds.append(N_d_best)
        N_ps.append(N_p_best)
        N_ss.append(N_s_best)
        SCs.append(SC_candidate_best)
        cases.append("k")

    if not any([case_a_accepted, case_b_accepted, case_c_accepted, case_defgh_accepted, case_i_accepted, case_j_accepted, case_k_accepted]):
        raise ValueError("No case is accepted")

    return N_ds, N_ps, N_ss, SCs, cases
        
    

In [ ]:
solution_phi_fixed(N = N_value,
                   s = s_value,
                   delta = delta_value,
                   phi = phi_value,
                   f = f_value,
                   E = E_value,
                   M = M_value,
                   Delta_d = -3,
                   F = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = 0),
                  )

In [ ]:
small_dim = 50
Delta_ds = np.linspace(-50, 48, small_dim)
Delta_p_mins = np.linspace(-50, 48, small_dim)

alpha = alpha

fig, ax = plt.subplots()

N_ds_all, N_ps_all, N_ss_all, SCs_all, cases_all, Delta_d_all, Delta_p_min_all = [], [], [], [], [], [], []

for Delta_d, Delta_p_min in tqdm(itertools.product(Delta_ds, Delta_p_mins), ncols=80, total=small_dim**2):
    N_ds, N_ps, N_ss, SCs, cases =\
        solution_phi_fixed(N = N_value,
                           s = s_value,
                           delta = delta_value,
                           phi = phi_value,
                           f = f_value,
                           E = E_value,
                           M = M_value,
                           Delta_d = Delta_d,
                           F = stats.uniform(scale=Delta_p_max_minus_min_value, loc=Delta_p_min),
                           #F = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = Delta_p_min),
                          )

    
    N_ds_all += N_ds
    N_ps_all += N_ps
    N_ss_all += N_ss
    SCs_all += SCs
    cases_all += cases
    Delta_d_all += [Delta_d for k in range(len(N_ds))]
    Delta_p_min_all += [Delta_p_min for k in range(len(N_ds))]


N_ds_all = np.array(N_ds_all)
N_ps_all = np.array(N_ps_all)
N_ss_all = np.array(N_ss_all)
SCs_all = np.array(SCs_all)
cases_all = np.array(cases_all)
Delta_d_all = np.array(Delta_d_all)
Delta_p_min_all = np.array(Delta_p_min_all)
colors_all = case_to_color_transform(cases_all)

ax.scatter(Delta_p_min_all, Delta_d_all, color=colors_all, alpha=alpha, s=10, linewidth=0)

ax.set_xlabel(r"$\Delta_p^{\text{min}}$")
ax.set_ylabel(r"$\Delta_d$")

    

In [ ]:
small_dim = 100

#N = 22900
N = 22950
Delta_p_min = -25
Delta_p_max_minus_min = 15
Delta_d = -50
F = stats.triang(c = 0.7, scale = Delta_p_max_minus_min_value, loc = Delta_p_min)


N_ps = np.linspace(0, N * M_value / (M_value + 1), small_dim)

def SC_last_term_vectorized(N_p):
    return N * integr.quad(lambda x: x * F.pdf(x), F.ppf(0), F.ppf(N_p / N))[0]
SC_last_term_vectorized = np.vectorize(SC_last_term_vectorized)
SCs_last = SC_last_term_vectorized(N_ps) # takes a very long time to compute, keep dim small.

N_ds = np.linspace(0, N, small_dim)
N_ds, N_ps = np.meshgrid(N_ds, N_ps)
to_delete, SCs_last = np.meshgrid(np.zeros(small_dim), SCs_last)
del to_delete
N_ss = N - N_ds - N_ps


SCs = delta_value / s_value * (N_ss**2 / (1 - phi_value) + N_ds * (N_ds + N_ps) / phi_value)\
        + (N - N_ps) * (f_value+E_value)\
        + N_ds * Delta_d\
        + SCs_last

SCs[N_ds < 0] = np.nan
SCs[N_ps < 0] = np.nan
SCs[N_ss < 0] = np.nan
SCs[N_ds * M_value < N_ps] = np.nan

N_ds2, N_ps2, N_ss2, SCs2, cases2 =\
        solution_phi_fixed(N = N,
                           s = s_value,
                           delta = delta_value,
                           phi = phi_value,
                           f = f_value,
                           E = E_value,
                           M = M_value,
                           Delta_d = Delta_d,
                           F = F
                          )

print(cases2)

SC_index = SCs2.index(min(SCs2))
SC_opt = SCs2[SC_index]
case_opt = cases2[SC_index]
N_d_opt = N_ds2[SC_index]
N_p_opt = N_ps2[SC_index]
N_s_opt = N_ss2[SC_index]

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.set_xlabel(r'$N_d$')
ax.set_ylabel(r'$N_p$')


ax.plot_surface(N_ds, N_ps, SCs, cmap="viridis", label="SC")
ax.scatter(N_d_opt, N_p_opt, SC_opt, color=color_legend[case_opt], s=30)
ax.view_init(-90, 38, 0)

In [ ]:
small_dim = 100
Ns = np.linspace(4.36055, 4.36065, small_dim)
#Ns = np.linspace(np.log(700) / np.log(10), np.log(1200) / np.log(10), small_dim)
Ns = 10**(Ns)

Delta_d = -50
Delta_p_min = -25
Delta_p_max_minus_min = 15

alpha = alpha
alpha_scatter = 0.4

fig, (ax_N_d, ax_N_p, ax_N_s, ax_N_d2, ax_N_p2, ax_N_s2, ax_SC) = plt.subplots(7, 1, tight_layout = True, figsize=(10, 10))
ax_N_d.set_xscale("log")
ax_N_p.set_xscale("log")
ax_N_s.set_xscale("log")
ax_N_d2.set_xscale("log")
ax_N_p2.set_xscale("log")
ax_N_s2.set_xscale("log")
ax_SC.set_xscale("log")

N_ds_selected, N_ps_selected, N_ss_selected, SCs_selected, cases_selected, Ns_selected = [], [], [], [], [], []
N_ds_all, N_ps_all, N_ss_all, SCs_all, cases_all, Ns_all = [], [], [], [], [], []

for N in tqdm(Ns, ncols=80):
    N_ds_out, N_ps_out, N_ss_out, SCs_out, cases_out =\
        solution_phi_fixed(N = N,
                           s = s_value,
                           delta = delta_value,
                           phi = phi_value,
                           f = f_value,
                           E = E_value,
                           M = M_value,
                           Delta_d = Delta_d,
                           #F = stats.uniform(scale=Delta_p_max_minus_min_value, loc=Delta_p_min)
                           F = stats.triang(c = 0.7, scale = Delta_p_max_minus_min, loc = Delta_p_min)
                          )

    SC_index_selected = SCs_out.index(min(SCs_out))
    SC_selected = SCs_out[SC_index_selected]
    case_selected = cases_out[SC_index_selected]
    N_d_selected = N_ds_out[SC_index_selected]
    N_p_selected = N_ps_out[SC_index_selected]
    N_s_selected = N_ss_out[SC_index_selected]

    N_ds_selected.append(N_d_selected)
    N_ps_selected.append(N_p_selected)
    N_ss_selected.append(N_s_selected)
    SCs_selected.append(SC_selected)
    cases_selected.append(case_selected)
    Ns_selected.append(N)

    N_ds_all += N_ds_out
    N_ps_all += N_ps_out
    N_ss_all += N_ss_out
    SCs_all += SCs_out
    cases_all += cases_out
    Ns_all += [N for k in range(len(N_ds_out))]

N_ds_selected = np.array(N_ds_selected)
N_ps_selected = np.array(N_ps_selected)
N_ss_selected = np.array(N_ss_selected)
SCs_selected = np.array(SCs_selected)
cases_selected = np.array(cases_selected)
Ns_selected = np.array(Ns_selected)
N_ds_all = np.array(N_ds_all)
N_ps_all = np.array(N_ps_all)
N_ss_all = np.array(N_ss_all)
SCs_all = np.array(SCs_all)
cases_all = np.array(cases_all)
Ns_all = np.array(Ns_all)

colors_selected = case_to_color_transform(cases_selected)
colors_all = case_to_color_transform(cases_all)

ax_N_d.plot(Ns_selected, N_ds_selected / Ns_selected, alpha=1, color="black", linewidth=1)
ax_N_d.scatter(Ns_all, N_ds_all / Ns_all, color=colors_all, alpha=alpha_scatter, s=13)
ax_N_d2.plot(Ns_selected, N_ds_selected, alpha=1, color="black", linewidth=1)
ax_N_d2.scatter(Ns_all, N_ds_all, color=colors_all, alpha=alpha_scatter, s=13)

ax_N_p.plot(Ns_selected, N_ps_selected / Ns_selected, alpha=1, color="black", linewidth=1)
ax_N_p.scatter(Ns_all, N_ps_all / Ns_all, color=colors_all, alpha=alpha_scatter, s=13)
ax_N_p2.plot(Ns_selected, N_ps_selected, alpha=1, color="black", linewidth=1)
ax_N_p2.scatter(Ns_all, N_ps_all, color=colors_all, alpha=alpha_scatter, s=13)

ax_N_s.plot(Ns_selected, N_ss_selected / Ns_selected, alpha=1, color="black", linewidth=1)
ax_N_s.scatter(Ns_all, N_ss_all / Ns_all, color=colors_all, alpha=alpha_scatter, s=13)
ax_N_s2.plot(Ns_selected, N_ss_selected, alpha=1, color="black", linewidth=1)
ax_N_s2.scatter(Ns_all, N_ss_all, color=colors_all, alpha=alpha_scatter, s=13)

ax_SC.plot(Ns_selected, SCs_selected, alpha=1, color="black", linewidth=1)
ax_SC.scatter(Ns_all, SCs_all, color=colors_all, alpha=alpha_scatter, s=13)


ax_N_d.set_ylim(bottom=0, top=1)
ax_N_p.set_ylim(bottom=0, top=1)
ax_N_s.set_ylim(bottom=0, top=1)

ax_SC.set_xlabel(r"$N$")

ax_N_d.set_ylabel(r"$\frac{N_d}{N}$")
ax_N_p.set_ylabel(r"$\frac{N_p}{N}$")
ax_N_s.set_ylabel(r"$\frac{N_s}{N}$")
ax_N_d2.set_ylabel(r"$N_d$")
ax_N_p2.set_ylabel(r"$N_p$")
ax_N_s2.set_ylabel(r"$N_s$")
ax_SC.set_ylabel(r"$SC$")

test